<a id="teteje"></a>
# 01 Adatelőkészítés: Data Preparation (dataset beszerzés, tisztítás és transzformáció)
---
**Függőség:** `config.py` (Útvonalak definíciója)

---

**Bemenet:** UCI ML Repository - Online Retail II (automatikus letöltés, nincs szükség kézi beavatkozásra, letölti az .xlsx fájlt)  
**Kimenetek:** 
- `data/raw/online_retail_II.xlsx` (letöltött nyers forrás, cache)
- `data/raw/online_retail_raw.parquet` (Nyers adat gyorsítótárazva)
- `data/processed/online_retail_cleaned.parquet` (Köztes tisztított állapot)
- `data/processed/online_retail_ready_for_rfm.parquet` **(Végső kimenet a következő fázishoz)**

---


## 0. Adatbetöltés és Parquet-konverzió

A nyers adathalmaz betöltése teljesen automatizált: a `0.1` cella letölti az UCI ML Repository-ból
az Online Retail II adathalmazt (id=502), kibontja a zip-ből az XLSX-et, majd a `0.2` cella
mindkét sheetből (2009–2010, 2010–2011) összefűzve közvetlenül Parquet formátumba konvertálja —
CSV közbenső lépés nélkül.

Mindkét cella **idempotens**: ha a fájl már létezik lemezen, a letöltés és a konverzió automatikusan
kimarad. Kézi beavatkozásra csak akkor van szükség, ha az UCI szervere nem elérhető
(ebben az esetben a cella részletes instrukciót ad a Kaggle-ről történő kézi letöltéshez).


In [ ]:
# ============================================================
# 0.1 – Konfiguráció, könyvtárak és adathalmaz letöltése
# ============================================================
import io
import zipfile
import urllib.request
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from config import (
    PROJECT_ROOT, RAW_DIR, PROCESSED_DIR, MODELS_DIR,
    RAW_FILE, RAW_XLSX, PARQUET_OUT, CLEANED_PARQUET, READY_FOR_RFM_PARQUET
)

# Mappastruktúra létrehozása (config.py is megteszi importkor, de explicit is egyértelműbb)
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"RAW_XLSX     : {RAW_XLSX}")
print(f"PARQUET_OUT  : {PARQUET_OUT}\n")

UCI_ZIP_URL = "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"

if PARQUET_OUT.exists():
    print(f"✅ Parquet már létezik, nincs szükség letöltésre: {PARQUET_OUT}")
elif RAW_XLSX.exists():
    print(f"✅ Nyers XLSX megtalálható, konverzió következik: {RAW_XLSX}")
else:
    print(f"⬇️  Adathalmaz letöltése az UCI ML Repository-ból ...")
    print(f"   URL: {UCI_ZIP_URL}")
    try:
        with urllib.request.urlopen(UCI_ZIP_URL, timeout=120) as response:
            zip_bytes = response.read()
        print(f"   Letöltve: {len(zip_bytes) / 1e6:.1f} MB")

        with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
            xlsx_names = [n for n in zf.namelist() if n.lower().endswith(".xlsx")]
            if not xlsx_names:
                raise FileNotFoundError(
                    f"Nem találtam XLSX-et a zip-ben. Tartalom: {zf.namelist()}"
                )
            target = max(xlsx_names, key=lambda n: zf.getinfo(n).file_size)
            print(f"   Kibontás: {target} → {RAW_XLSX.name}")
            RAW_XLSX.write_bytes(zf.read(target))

        print(f"\n✅ Nyers XLSX mentve: {RAW_XLSX}  ({RAW_XLSX.stat().st_size / 1e6:.1f} MB)")

    except Exception as exc:
        print(f"\n❌ Automatikus letöltés sikertelen: {exc}")
        print(
            "\nKézi alternatíva (Kaggle):\n"
            "  1. Töltsd le: https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci/data\n"
            "  2. Csomagold ki az 'online_retail_II.xlsx'-et ide: " + str(RAW_XLSX) + "\n"
            "  3. Futtasd újra ezt a cellát!"
        )
        raise


In [ ]:
# ============================================================
# 0.2 – XLSX → Parquet konverzió (python-calamine engine)
# ============================================================
# Az openpyxl pure Python cella-per-cella olvasás helyett a calamine
# Rust-alapú parser 10-30x gyorsabb ugyanazon XLSX fájlon.
# Telepítés: pip install python-calamine  (requirements.txt-be is felkerült)

if PARQUET_OUT.exists():
    print(f"✅ Parquet már létezik, kihagyjuk a konverziót: {PARQUET_OUT}")
else:
    if RAW_XLSX.exists():
        import time
        print(f"XLSX betöltése (calamine engine): {RAW_XLSX}")
        t0 = time.time()

        # sheet_name=None → minden sheet dict-ként; calamine-nél dtype nem támogatott,
        # a típusokat konverzió után állítjuk be
        sheet_names = pd.ExcelFile(RAW_XLSX, engine="calamine").sheet_names
        frames = []
        for name in sheet_names:
            df_sheet = pd.read_excel(RAW_XLSX, sheet_name=name, engine="calamine")
            print(f"  • Sheet '{name}': {len(df_sheet):,} sor  ({time.time()-t0:.1f}s)")
            frames.append(df_sheet)
        df_raw = pd.concat(frames, ignore_index=True)

        # --- Típusok beállítása betöltés után ---
        df_raw["InvoiceDate"] = pd.to_datetime(df_raw["InvoiceDate"])
        df_raw["Customer ID"] = df_raw["Customer ID"].astype("Int64")
        for col in ("Invoice", "StockCode", "Description", "Country"):
            df_raw[col] = df_raw[col].astype("string")
        df_raw["Quantity"] = df_raw["Quantity"].astype("float64")
        df_raw["Price"]    = df_raw["Price"].astype("float64")

    elif RAW_FILE.exists():
        #! CSV fallback – visszafelé kompatibilitás kézi letöltésnél
        import time; t0 = time.time()
        print(f"CSV fallback betöltése: {RAW_FILE}")
        df_raw = pd.read_csv(
            RAW_FILE,
            dtype={
                "Invoice": "string", "StockCode": "string", "Description": "string",
                "Quantity": "float64", "Price": "float64",
                "Customer ID": "float64", "Country": "string",
            },
            parse_dates=["InvoiceDate"],
            encoding="utf-8",
        )
        df_raw["Customer ID"] = df_raw["Customer ID"].astype("Int64")
    else:
        raise FileNotFoundError(
            f"Sem XLSX ({RAW_XLSX}), sem CSV ({RAW_FILE}) nem található!\n"
            "Futtasd újra a 0.1-es cellát a letöltéshez."
        )

    # --- Parquet mentés ---
    df_raw.to_parquet(PARQUET_OUT, compression="snappy", index=False)

    elapsed = time.time() - t0
    size_mb  = PARQUET_OUT.stat().st_size / 1_048_576
    print(f"\n✅ Parquet mentve: {PARQUET_OUT}")
    print(f"   Fájlméret:  {size_mb:.1f} MB")
    print(f"   Sorok:      {len(df_raw):,} | Oszlopok: {df_raw.shape[1]}")
    print(f"   Futási idő: {elapsed:.1f}s")
    print(f"\nSéma:\n{df_raw.dtypes}")


## 1. Adattisztítás

### 1.1. Első lépések

A Customer Lifetime Value (CLV) modellezés és az RFM szegmentáció alapfeltétele, hogy az adatokat vásárlói szinten tudjuk aggregálni. Ennek megfelelően az alábbi tisztítási lépéseket végezzük el:

- **Anonim tranzakciók eltávolítása:** A hiányzó `Customer ID`-val rendelkező sorok kötelezően eldobandók.
- **Azonosítók típuskonverziója:** A `Customer ID`-t numerikus formátumból (Int64) string (object) típusra alakítjuk, hogy elkerüljük az EDA során a fals statisztikákat, és megelőzzük a későbbi gépi tanulási modelleknél (K-means, XGBoost) az adatszivárgást (data leakage).
- **Érvénytelen értékek szűrése:** A 0 vagy negatív `Price` értékkel rendelkező sorok eltávolítása.
- **Duplikátumok szűrése:** Az azonos tartalmú, ismétlődő sorok törlése.
- **Adminisztratív és nem termék kódok eltávolítása:** A nyers adatokon végzett előzetes **SQLite-os adatfeltárás** során kiderült, hogy az adathalmazban jelentős számú, torzító hatású adminisztratív bejegyzés található (pl. `POST`, `M`, `BANK CHARGES`, `AMAZONFEE`). Ezek mellé soroljuk a `C2` (Carriage) kódot is, amely egy kiugróan magas árú, fix tengerentúli fuvardíjra utal. Mivel ezek nem kézzelfogható termékek (és gyakran extrém negatív korrekciós értékeket képviselnek), teljesen kiszűrjük őket, hogy ne torzítsák a termékalapú kosárelemzést és a CLV-t.
- **Fejlesztői teszt bejegyzések szűrése:** Szintén az SQLite-os lekérdezések buktatták le a rendszerben maradt teszt tranzakciókat (pl. `TEST001`, `TEST002` cikkszámok, vagy `test` leírások). Mivel ezek irreális árakkal és mennyiségekkel rontanák a statisztikákat, a `StockCode` és a `Description` oszlopok együttes vizsgálatával az összeset eldobjuk.

*Megjegyzés: A sztornó/visszaküldött tételeket ('C' prefix az Invoice oszlopban) ebben a fázisban szándékosan megtartjuk, mivel ezek a későbbiekben negatív feature-ként szolgálnak a vásárlói érték (CLV) számításánál.*

In [ ]:
# ============================================================
# 1.1 Adattisztítás
# ============================================================

print("Adattisztítás megkezdése...\n")

# 1.1.1. Nyers Parquet betöltése
df = pd.read_parquet(PARQUET_OUT)
eredeti_sorok_szama = len(df)
print(f"Kiinduló sorok száma: {eredeti_sorok_szama:,}")
print("-" * 50)

# 1.1.2. Hiányzó Customer ID-k eldobása
df = df.dropna(subset=["Customer ID"])
id_nelkuli_sorok = eredeti_sorok_szama - len(df)
print(f"✔️ Nincsenek anonim vásárlók. (Eldobva: {id_nelkuli_sorok:,} sor)")

# Opcionális, de ajánlott: Customer ID stringgé alakítása
df["Customer ID"] = df["Customer ID"].astype(str)
print("✔️ A Customer ID biztonságos string formátumban van.")

# 1.1.3. Érvénytelen árak szűrése (Price <= 0)
ervenytelen_ar_maszk = df["Price"] <= 0
ervenytelen_ar_sorok = ervenytelen_ar_maszk.sum()
df = df[~ervenytelen_ar_maszk]
print(f"✔️ Az érvénytelen (0 vagy negatív) árak eltűntek. (Eldobva: {ervenytelen_ar_sorok:,} sor)")

# --- Sztornók megtartása ---
print("✔️ A sztornók szándékosan megmaradnak a feature engineeringhez (Quantity <= 0 szándékosan nincs szűrve).")

# 1.1.4. Nem termék jellegű tranzakciók kiszűrése
nem_termek_maszk = (
    df["StockCode"].astype(str).str.replace(" ", "").str.isalpha() | 
    (df["StockCode"].astype(str).str.upper() == "C2")
)
nem_termek_sorok = nem_termek_maszk.sum()
df = df[~nem_termek_maszk]
print(f"✔️ A 'BANK CHARGES' jellegű és 'C2' fuvardíj kódok célzottan eldobásra kerültek. (Eldobva: {nem_termek_sorok:,} sor)")

# 1.1.5. Teszt bejegyzések eltávolítása
teszt_maszk = (
    df["Description"].astype(str).str.contains("TEST", case=False, na=False) |
    df["StockCode"].astype(str).str.contains("TEST", case=False, na=False)
)
teszt_sorok = teszt_maszk.sum()
df = df[~teszt_maszk]
print(f"✔️ A rejtett, üres leírású tesztkódok is fennakadtak a szűrőn. (Eldobva: {teszt_sorok:,} sor)")

# 1.1.6. Duplikátumok eldobása
duplikatum_maszk = df.duplicated()
duplikatum_sorok = duplikatum_maszk.sum()
df = df[~duplikatum_maszk]
print(f"✔️ A duplikátumok eltávolítva. (Eldobva: {duplikatum_sorok:,} sor)")

# 1.1.7. Tisztított adatok mentése checkpointként
df.to_parquet(CLEANED_PARQUET, compression="snappy", index=False)

vegleges_sorok_szama = len(df)
print("-" * 50)
print(f"Adattisztítás eredménye:")
print(f"Megmaradt tiszta sorok száma: {vegleges_sorok_szama:,} "
      f"(az eredeti {vegleges_sorok_szama/eredeti_sorok_szama*100:.1f}%-a)")
print(f"Tisztított adatok mentve: {CLEANED_PARQUET}")

### 1.2. Adatszerkezet és típusok ellenőrzése
A tisztítási lépések befejeztével elengedhetetlen ellenőrizni az adathalmaz technikai állapotát. A `df.info()` kimenetével az alábbiakat validáljuk a Feature Engineering megkezdése előtt:
* **Nincsenek hiányzó értékek:** Minden oszlopban hiánytalan a `Non-Null Count` (az "anonim" vásárlásokat sikeresen kidobtuk).
* **Megfelelő adattípusok:** Az `InvoiceDate` helyesen `datetime` típusú, a `Customer ID` pedig konzisztens szöveges (string/object) formátumú.
* **Memóriahatékonyság:** A felesleges és hibás sorok eltávolításával optimalizáltuk a memóriahasználatot, ami a későbbi gépi tanulási modellek (K-means, XGBoost) illesztésénél kritikus lesz.

In [ ]:
# ============================================================
# 1.2 "QA" ellenőrzés 1/2
# ============================================================
# Azt validálja, hogy az anonimok és érvénytelen árak ki letek-e dobva, és a típusok helyesek-e
df.info()

### 1.3. Technikai outlierek és anomáliák szűrése (Feature Engineering előtt)

Mielőtt rátérnénk a vásárlói érték (RFM és CLV) kiszámítására, el kell távolítanunk a rendszerben maradt **technikai anomáliákat**. Fontos: itt még nem a statisztikai outliereket (pl. a kiugróan sokat költő VIP vevőket) kezeljük, hanem azokat a tranzakciókat, amelyek **torzítanák vagy logikailag ellehetetlenítenék** a vásárlói profilalkotást.

Ebben a lépésben az alábbi "zajokat" szűrjük:
1. **"Árva" visszaküldések és negatív élettartam-értékű (CLV) ügyfelek:** Előfordulnak olyan ügyfelek, akiknek a vizsgált időszakban több a sztornója, mint a vásárlása (tehát a nettó költésük $\le 0$). Ennek tipikus oka, hogy a tényleges vásárlás az adathalmaz kezdete (2009. dec. 1.) *előtt* történt. Mivel róluk nem tudunk valós vásárlói értéket (Monetary) számolni, az ilyen ügyfelek összes sorát kötelezően eldobjuk.
2. **Fizikailag lehetetlen tranzakciók (Extrém Quantity):** Az adatok előzetes vizsgálata kimutatta, hogy léteznek +/- 80 000 darabos tételek. Egy ajándéktárgy-nagykereskedésben (B2B) a pár ezer darabos rendelés még reális, de a 10 000 feletti kiugrások szinte kivétel nélkül azonnal sztornózott rendszerhibák vagy adminisztrációs tesztek. Ezeket a sorokat eltávolítjuk, hogy ne torzítsák a kosárméret-statisztikákat.

In [ ]:
# ============================================================
# 1.3 - Technikai outlierek, "árva" sztornók és hibák szűrése
# ============================================================
from config import Q_THRESHOLD

df = pd.read_parquet(CLEANED_PARQUET)
print("Technikai outlierek és anomáliák szűrése...")
eredeti_sorok = len(df)

# 1.4.1. Tranzakció nettó értékének (LineTotal) kiszámítása
df['LineTotal'] = df['Quantity'] * df['Price']

# 1.4.2. Manuális korrekciók ("Adjustment") szűrése a Description alapján
# A felfedező elemzés során találtunk "Adjustment by Peter..." jellegű sorokat
adjustment_mask = df['Description'].str.contains('ADJUSTMENT', case=False, na=False)
df = df[~adjustment_mask]

# 1.4.3. Ügyfélszintű aggregáció: mekkora az adott ügyfél teljes nettó költése az időszakban?
customer_revenue = df.groupby('Customer ID')['LineTotal'].sum().reset_index()

# Azonosítjuk azokat az ügyfeleket, akiknek a teljes egyenlege nulla vagy negatív
invalid_customers = customer_revenue[customer_revenue['LineTotal'] < 0]['Customer ID']
df = df[~df['Customer ID'].isin(invalid_customers)]

# 1.4.4. Extrém darabszámok (Quantity) szűrése (pl. +/- 80 ezer darabos rendszerhibák)
extreme_quantity_mask = df['Quantity'].abs() > Q_THRESHOLD
df = df[~extreme_quantity_mask]

vegleges_sorok = len(df)
print(f"✔️ Manuális korrekciók, szigorúan negatív egyenlegű ügyfelek (< 0) és >{Q_THRESHOLD} db-os anomáliák eltávolítva.")
print(f"Elemzésre (RFM) kész sorok száma: {vegleges_sorok:,} "
      f"(a kiinduló tisztított adathalmaz kb. {vegleges_sorok/eredeti_sorok*100:.1f}%-a megmaradt)\n")

# Tisztított, RFM-re kész adatok mentése memóriahatékony Parquet formátumban
df.to_parquet(READY_FOR_RFM_PARQUET, compression="snappy", index=False)
print(f"Adathalmaz mentve a következő fázishoz: {READY_FOR_RFM_PARQUET}")

### 1.4 Statisztikai érvényesség és üzleti logika ellenőrzése
Míg a kezdeti EDA során a `describe()` a hibák felfedezését szolgálta, itt már **minőségbiztosítási (QA)** szerepe van. Az alábbi kimenet bizonyítja, hogy a tisztítási logikánk sikeres volt:
* **Érvényes árak:** A `Price` oszlop minimum értéke szigorúan **> 0** (nincsenek ingyenes vagy negatív árú, adminisztratív tételek).
* **Reális mennyiségek:** A `Quantity` oszlop extrém, rendszerhibából fakadó kilengései (pl. +/- 80 000 darab) eltűntek, a maximum és minimum értékek a beállított küszöbértéken belül maradtak.
* **Adatszivárgás megelőzése:** Nincsenek olyan anomáliák, amik a későbbi aggregált RFM metrikákat (pl. átlagos kosárérték) irreális irányba torzítanák.

In [ ]:
# ============================================================
# 1.4 "QA" ellenőrzés 2/2
# ============================================================
df.describe()

### 1.5. Cutoff validáció
**Elegendő adat van-e a célablakban?**

Mielőtt a `config.py`-ban rögzített `CUTOFF_DATE` értékét véglegesnek tekintjük,
ellenőrizzük, hogy a vágási pont után marad-e elegendő tranzakció és ügyfél
a célváltozó (y) majdani legyártásához.

Az ábra a teljes időszak napi bevételét mutatja; a piros szaggatott vonal jelöli
a tervezett cutoff-ot. Ha utána az adatok sűrűsége és volumene elegendőnek tűnik,
a `CUTOFF_DATE` értéke végleges, a `03_churn_prediction.ipynb` notebook már ezt
az értéket veszi alapul.

In [ ]:
# ============================================================
# 1.5. Cutoff validáció
# ============================================================

import matplotlib.pyplot as plt
from config import CUTOFF_DATE

# Dinamikus dátum beolvasása a configból
cutoff_date = pd.to_datetime(CUTOFF_DATE)

# Nézzük meg a tranzakciók napi eloszlását
daily_revenue = df.groupby(df['InvoiceDate'].dt.date)['LineTotal'].sum()

plt.figure(figsize=(15, 5))
daily_revenue.plot(title="Napi bevétel a vizsgált időszakban", color="#1f77b4")
# Itt is a dinamikus változót használjuk
plt.axvline(cutoff_date, color='red', linestyle='--', label="Tervezett Cutoff (y ablak kezdete)")
plt.ylabel("Nettó Bevétel (GBP)")
plt.legend()
plt.grid(True, alpha=0.3)

# Mennyi adat (bevétel és tranzakció) esik a célablakba?
target_window = df[df['InvoiceDate'] >= cutoff_date]

print(f"A célablakba eső tranzakciósorok száma: {len(target_window):,}")
print(f"A célablakba eső nettó bevétel: £ {target_window['LineTotal'].sum():,.2f}")
print(f"Egyedi vásárlók a célablakban: {target_window['Customer ID'].nunique():,}")

<div align="center">
  <br>
  <a href="#teteje">
    <img src="https://img.shields.io/badge/%E2%AC%86%20Vissza%20a%20tetej%C3%A9re-c0253f?style=for-the-badge" alt="Vissza a tetejére" width="250">
  </a>
  <br>
</div>

*Az ugrás gomb nem minden környezetben működik!

# Dokumentáció frissítése README.md-ben és docs mappában
🚨 **Ctrl+S szükséges az alábbi cella futtatása előtt, mivel az nbconvert lemezről olvas!**

In [ ]:
# 03-as notebook docs generálása/frissítése
!python update_docs.py --notebook 01_data_preparation.ipynb